# TD 2 : cadrer un problème, et la chasse à la fuite

## Comment se passe cette journée

Vous travaillez **en binôme**, et vous changez de clavier à chaque étape. Celui qui ne tape pas lit l'énoncé à voix haute et vérifie.

**Personne ne lève la main pour savoir s'il a juste.** Chaque exercice se termine par une cellule de vérification qui vous le dit, et qui explique ce qui ne va pas si ce n'est pas bon. Les indices sont repliés sous chaque exercice : ouvrez-les plutôt que d'attendre.

**Quand vous finissez une étape**, marquez-la sur le tableau au mur. C'est ce qui permet de voir où en est la salle sans faire le tour des rangs.

## Ce que vous saurez faire

1. transformer une demande en spécification : target, maille, metric
2. charger un fichier européen sans perdre la moitié des colonnes
3. repérer un marqueur de valeur manquante que personne ne vous a signalé
4. distinguer une colonne dérivée d'une colonne informative
5. construire un baseline, puis détecter une fuite qui améliore le score

## Prérequis

- avoir fini le TD 1
- avoir suivi le bloc B de la journée 1

**Durée :** 100 minutes

## Le contexte

Une **agence de santé environnementale** surveille la pollution de l'air. L'analyseur certifié coûte cher à exploiter. La question posée est simple : est-ce que les capteurs bon marché peuvent dire ce que l'analyseur certifié aurait dit ? Les données sont celles d'une station réelle, sur une année.

## Le déroulé

| quand | quoi |
|---|---|
| 2:00 | étape 1 : de la demande à la spécification |
| 2:20 | étape 2 : charger, et ne pas croire le fichier |
| 2:50 | étape 3 : profiler, et le -200 |
| 3:10 | étape 4 : baseline, modèle, et la fuite |
| 3:45 | déjeuner |

In [ ]:
# Amorce : a executer en premier, une seule fois par notebook.
import sys, pathlib
racine = pathlib.Path.cwd()
while not (racine / "aeolis").is_dir() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine))
import os
os.chdir(racine)

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aeolis import plots, evaluate as ev

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Les donnees sont dans cours_fr/data pendant le cours, et dans donnees/ dans le
# dossier qui vous est remis. On prend celui qui existe, pour que le meme notebook
# fonctionne dans les deux cas.
DONNEES = next((c for c in (racine / "cours_fr" / "data", racine / "donnees",
                            racine / "data") if c.is_dir()), racine)

print("racine du cours :", racine.name)
print("donnees :", DONNEES.name, "->",
      sorted(p.name for p in DONNEES.glob("*.parquet")) or "aucun parquet")
print("pret")

---

## Étape 1 : De la demande à la spécification

**20 minutes.** Une demande n'est pas un problème. Cinq éléments la transforment, et le cinquième ne vous appartient pas.

Trois demandes arrivent, telles quelles. Pour chacune : que prédit-on exactement, sur quelle maille, et comment mesure-t-on le succès ?

**Deux ou trois lignes par demande.** Là où la demande est réellement ambiguë, écrivez ce que vous iriez demander. Dire « cette question est mal posée, voilà ce qu'il me manque » est une bonne réponse, pas un aveu.

In [ ]:
# En binome, 12 minutes. Ecrivez dans les chaines.
demande_1 = "Nos capteurs derivent. Peut-on savoir quand les recalibrer ?"
spec_1 = '...'

demande_2 = "La pollution semble pire certains jours. Pourquoi ?"
spec_2 = '...'

demande_3 = "Peut-on arreter de payer l'analyseur certifie ?"
spec_3 = '...'

**Indice 1.** Pour chacune, posez-vous les cinq questions dans l'ordre : quelle target, quelle maille, quels inputs disponibles, quelle metric, et à partir de quand c'est assez bon.

**Indice 2.** La deuxième n'est pas un problème de prédiction. On y demande une explication, et une explication ne se traite pas comme une prévision.

In [ ]:
from aeolis.checks import done
done(spec_1, spec_2, spec_3)
for nom, s in [('spec_1', spec_1), ('spec_2', spec_2), ('spec_3', spec_3)]:
    assert len(s) > 120, f'{nom} fait {len(s)} caracteres, c est trop court'
print('Comparez avec la solution. La note porte sur trois choses :')
print('  - avoir nomme une target, une maille et une metric pour 1 et 3')
print('  - avoir dit que la 2 n est pas un probleme de prediction')
print('  - avoir ecrit ce que vous iriez DEMANDER, au lieu de l inventer')

---

## Étape 2 : Charger, et ne pas croire le fichier

**30 minutes.** Le fichier vient d'un export européen. Il va résister.

## Mémo de code

Chaque exercice ci-dessous se résout en adaptant un de ces patrons. Copiez, changez les noms, exécutez. Ce n'est pas tricher : c'est ainsi que l'on écrit du code de data.

```python
# lire un CSV europeen : les DEUX arguments comptent
table = pd.read_csv('data/station_raw.csv', sep=';', decimal=',')

# le type réel de chaque colonne, AVANT toute statistique
table.dtypes

# supprimer les lignes et colonnes entierement vides
table = table.dropna(how='all').dropna(axis=1, how='all')

# remplacer un sentinel par un vrai marqueur de valeur manquante
table = table.replace(-200, np.nan)

# la part de valeurs manquantes par colonne
table.isna().mean()

# supprimer une colonne
table = table.drop(columns=['NMHC(GT)'])

# ne garder que les lignes qui ont une target
table = table.dropna(subset=['CO(GT)'])
```

### 2.1 Le premier essai échoue, et il faut le voir échouer

In [ ]:
# A FAIRE : chargez le fichier NAIVEMENT, sans aucun argument,
#           et regardez ce que vous obtenez.
naif = ...

print('forme :', naif.shape)
print('colonnes :', list(naif.columns)[:3])

**Indice 1.** `pd.read_csv('data/station_raw.csv')`, sans rien d'autre. Le but est de voir le problème, pas de l'éviter.

**Indice 2.** `naif = pd.read_csv('data/station_raw.csv')`

In [ ]:
assert naif.shape[1] == 1, (
    f'attendu une seule colonne, obtenu {naif.shape[1]}')
print('Une seule colonne. Le fichier utilise le point-virgule comme separateur,')
print('et pandas a tout mis dans une colonne unique sans se plaindre.')
print('Aucune erreur, aucun avertissement. Juste un resultat faux.')

### 2.2 La version qui marche, et le réflexe qui va avec

In [ ]:
# A FAIRE (1) : charger correctement. DEUX arguments sont necessaires.
station = ...

# A FAIRE (2) : afficher le type réel de chaque colonne.
...

print('forme :', station.shape)

**Indice 1.** Le séparateur est `;`. Et le marqueur décimal est la virgule, pas le point : sans cela, les nombres arrivent en texte.

**Indice 2.** `pd.read_csv('data/station_raw.csv', sep=';', decimal=',')` puis `print(station.dtypes)`

In [ ]:
assert station.shape[1] == 17, (
    f'attendu 17 colonnes, obtenu {station.shape[1]}. '
    'Avez-vous bien passe sep et decimal ?')
numeriques = (station.dtypes != object).sum()
assert numeriques >= 13, (
    f'seulement {numeriques} colonnes numeriques. Sans decimal=\',\' les '
    'nombres arrivent en texte et describe() les ignore en silence.')
print(f'correct : {station.shape[0]} lignes, {station.shape[1]} colonnes,')
print(f'dont {numeriques} numeriques.')
print('Deux colonnes sont entierement vides : ce sont les point-virgules')
print('en fin de ligne. Elles partent a l etape suivante.')

### 2.3 Deux colonnes et 114 lignes qui n'existent pas

In [ ]:
# A FAIRE : supprimer les lignes ET les colonnes entierement vides.
propre = ...

print('avant :', station.shape)
print('apres :', propre.shape)

**Indice 1.** Le patron du mémo enchaîne deux `dropna`, l'un sur les lignes et l'autre sur les colonnes, avec `how='all'` pour ne supprimer que ce qui est entièrement vide.

**Indice 2.** `propre = station.dropna(how='all').dropna(axis=1, how='all')`

In [ ]:
assert propre.shape == (9357, 15), (
    f'attendu (9357, 15), obtenu {propre.shape}')
print('correct : 9 357 lignes et 15 colonnes.')
print('114 lignes vides en fin de fichier, 2 colonnes vides sur les cotes.')
print('Personne ne les a mises la expres : c est un export de tableur.')

---

## Étape 3 : Profiler, et le -200

**20 minutes.** Le moment le plus utile de la matinée. Un fichier peut n'avoir aucune valeur manquante et être plein de trous.

### 3.1 Combien de valeurs manquantes ?

In [ ]:
# A FAIRE (1) : combien de valeurs manquantes dans `propre` ?
manquantes = ...

# A FAIRE (2) : afficher describe(), et regarder la ligne min.
...

print(f'valeurs manquantes : {manquantes}')

**Indice 1.** `isna().sum().sum()` pour (1). Pour (2), `describe()` et regardez la ligne `min` colonne par colonne. Lisez-la avant de conclure.

**Indice 2.** `manquantes = propre.isna().sum().sum()` puis `display(propre.describe())`

In [ ]:
assert manquantes == 0, (
    f'attendu 0, obtenu {manquantes}')
mini = propre.select_dtypes('number').min()
print('Zero valeur manquante. Et maintenant la ligne min :')
print(mini.to_string())
assert (mini == -200).sum() >= 10, 'au moins dix colonnes descendent a -200'
print()
print('Chaque colonne de mesure descend a exactement -200,00.')
print('Une concentration negative n existe pas. -200 est un SENTINEL :')
print('la facon dont cet instrument ecrit pas de mesure.')

### 3.2 Remplacer le sentinel, et regarder la vérité

In [ ]:
# A FAIRE : remplacer -200 par un vrai marqueur de valeur manquante,
#           puis afficher la part manquante par colonne, triee.
vrai = ...
part_manquante = ...

print(part_manquante.head(6))

**Indice 1.** `replace(-200, np.nan)` pour la première partie. Puis `isna().mean()` et un tri décroissant.

**Indice 2.** `vrai = propre.replace(-200, np.nan)` puis `part_manquante = vrai.isna().mean().sort_values(ascending=False)`

In [ ]:
pire = part_manquante.index[0]
assert pire == 'NMHC(GT)', f"la pire colonne est NMHC(GT), vous trouvez {pire}"
assert part_manquante.iloc[0] > 0.9, 'NMHC(GT) est vide a plus de 90 %'
print(f'correct. La pire est {pire}, manquante a '
      f'{part_manquante.iloc[0]:.1%}.')
print('Elle est inutilisable et elle part. Les autres se traitent.')
print('Et la photo est maintenant honnete : avant le replace, vous auriez')
print('calcule des moyennes sur des -200 sans le savoir.')

---

## Étape 4 : Baseline, modèle, et la fuite

**30 minutes.** La partie évaluée. Le meilleur score du tableau sera celui qu'il faut refuser.

### 4.1 Nettoyer pour de bon, et choisir les colonnes

Deux familles de colonnes, et le choix entre elles **définit le projet** :

- les **capteurs bon marché** : `PT08.S1(CO)` à `PT08.S5(O3)`, plus `T`, `RH`, `AH`
- les **canaux de l'analyseur certifié** : `NOx(GT)`, `NO2(GT)`, `C6H6(GT)`

La target est `CO(GT)`, la mesure de l'analyseur certifié.

In [ ]:
# A FAIRE : NMHC(GT) part, et on ne garde que les lignes qui ont une target.
jeu = ...

print('lignes exploitables :', len(jeu))

**Indice 1.** Les deux patrons du mémo : `drop(columns=[...])` puis `dropna(subset=[...])` sur la target `CO(GT)`.

**Indice 2.** `jeu = vrai.drop(columns=['NMHC(GT)']).dropna(subset=['CO(GT)'])`

In [ ]:
assert 'NMHC(GT)' not in jeu.columns, 'NMHC(GT) devait partir'
assert jeu['CO(GT)'].isna().sum() == 0, 'il reste des lignes sans target'
assert len(jeu) > 7000, f'seulement {len(jeu)} lignes, c est trop peu'
print(f'correct : {len(jeu)} lignes exploitables.')

### 4.2 Le baseline, avant tout modèle

**Que donne la solution la plus bête ?** Sans cette réponse, aucun score ne veut rien dire.

In [ ]:
# A FAIRE : predire toujours la moyenne, et mesurer l erreur absolue moyenne.
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

CAPTEURS = ['PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)',
            'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']
REFERENCE = ['NOx(GT)', 'NO2(GT)', 'C6H6(GT)']
TARGET = 'CO(GT)'

def construire_xy(colonnes):
    sous = jeu[colonnes + [TARGET]].dropna()
    return sous[colonnes].values, sous[TARGET].values

X, y = construire_xy(CAPTEURS)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

mae_baseline = ...

print(f'BASELINE  MAE {mae_baseline:.3f}')

**Indice 1.** `DummyRegressor(strategy='mean')` s'entraîne et prédit comme n'importe quel modèle sklearn. Puis `mean_absolute_error`.

**Indice 2.** `bidon = DummyRegressor(strategy='mean').fit(X_tr, y_tr)` puis `mae_baseline = mean_absolute_error(y_te, bidon.predict(X_te))`

In [ ]:
assert abs(mae_baseline - 1.107) < 0.05, (
    f'attendu environ 1,107, obtenu {mae_baseline:.3f}')
print(f'correct : MAE {mae_baseline:.3f} mg/m3 en predisant toujours la moyenne.')
print('Tout modele qui ne fait pas mieux que ca ne sert a rien.')

### 4.3 Le modèle honnête

In [ ]:
# A FAIRE : un HistGradientBoostingRegressor sur les CAPTEURS seuls.
from sklearn.ensemble import HistGradientBoostingRegressor

honnete = ...
mae_honnete = ...

print(f'HONNETE   MAE {mae_honnete:.3f}')
print(f'soit {1 - mae_honnete / mae_baseline:.0%} de mieux que le baseline')

**Indice 1.** Même schéma que le baseline : `.fit(X_tr, y_tr)` puis `mean_absolute_error(y_te, modele.predict(X_te))`.

**Indice 2.** `honnete = HistGradientBoostingRegressor(random_state=42).fit(X_tr, y_tr)`

In [ ]:
assert abs(mae_honnete - 0.308) < 0.05, (
    f'attendu environ 0,308, obtenu {mae_honnete:.3f}')
assert mae_honnete < mae_baseline, 'le modele doit battre le baseline'
print(f'correct : MAE {mae_honnete:.3f}, soit '
      f'{1 - mae_honnete / mae_baseline:.0%} de mieux.')
print('Ce modele est deployable : il n utilise que les capteurs bon marche.')

### 4.4 La chasse à la fuite

**Prédisez avant de lancer.** On ajoute les trois canaux de l'analyseur certifié aux entrées. Que va faire le score ?

In [ ]:
# A FAIRE : le meme modele, les memes reglages, mais avec
#           CAPTEURS + REFERENCE en entree.
Xf, yf = ...
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.25,
                                             random_state=42)
mae_fuite = ...

print(f'AVEC FUITE  MAE {mae_fuite:.3f}')
print(f'soit {1 - mae_fuite / mae_honnete:.0%} de mieux que le modele honnete')

**Indice 1.** `construire_xy(CAPTEURS + REFERENCE)` donne le nouveau jeu. Le reste est identique à 4.3.

**Indice 2.** `Xf, yf = construire_xy(CAPTEURS + REFERENCE)` puis le même HistGradientBoostingRegressor.

In [ ]:
assert abs(mae_fuite - 0.259) < 0.05, (
    f'attendu environ 0,259, obtenu {mae_fuite:.3f}')
assert mae_fuite < mae_honnete, (
    'le modele avec fuite doit avoir l air MEILLEUR. C est tout le probleme.')
print(f'Le modele avec fuite obtient {mae_fuite:.3f} contre '
      f'{mae_honnete:.3f}.')
print()
print('Rien n est casse. Pas de bug, pas de donnees de test touchees,')
print('pas de melange. Le split est correct, le score est bien calcule.')
print('Et ce modele ne pourra JAMAIS etre mis en service, parce qu il')
print('utilise l instrument que le projet veut arreter de payer.')

## Ce que cette étape a établi

- Une demande n'est pas un problème : target, maille, inputs, metric, seuil.
- Un fichier européen a besoin de `sep=';'` et `decimal=','`, et il échoue en silence sans eux.
- 9 357 lignes et 15 colonnes, après avoir retiré 114 lignes et 2 colonnes entièrement vides.
- Zéro valeur manquante, et pourtant chaque colonne descend à -200. Lisez la ligne `min`.
- Baseline 1,107, modèle honnête 0,308, modèle avec fuite 0,259. Le meilleur des trois est celui qu'il faut refuser.

## Les erreurs classiques

- **Lancer `describe()` sans avoir vérifié `dtypes`.** Les colonnes texte sont ignorées sans un mot, et vous ne le saurez jamais.
- **Croire `isna()` sur un fichier d'instrument.** Lisez la ligne `min`. Un sentinel est invisible pour pandas.
- **Prendre le meilleur score du tableau.** Demandez d'abord si chaque colonne existera au moment de la prédiction.

## Pour aller plus loin, si vous avez fini en avance

- Reprendre 4.4 en ne gardant qu'un seul des trois canaux de référence. Combien d'avance obtient-on avec un seul ? La fuite est-elle proportionnelle ?
- La colonne `AH` se reconstruit à partir de `T` et `RH` avec un R2 de 0,88. Vérifiez-le, et demandez-vous si c'est une fuite ou autre chose.